# CQLite Python Bindings - Acceptance Testing

This notebook provides a comprehensive tour of the CQLite Python bindings (Epic #321 - M4).

**Prerequisites:**
- CQLite bindings installed (`maturin develop` in bindings/python/)
- Test data available (`bash test-data/scripts/fetch-datasets.sh`)
- Environment variable: `CQLITE_DATASETS_ROOT` pointing to test data

## 1. Environment Setup & Import Validation

In [1]:
import os
from pathlib import Path

# Import cqlite
import cqlite

# Verify version
print(f"CQLite version: {cqlite.version()}")
print(f"Module version: {cqlite.__version__}")

# Verify all public exports are available
expected_exports = [
    'open', 'Database', 'QueryResult', 'Row', 'ColumnInfo',
    'StreamingIterator', 'StreamingConfig', 'PreparedStatement', 'DatabaseStats',
    'CqliteError', 'SchemaError', 'QueryError', 'ParseError',
    'memory_optimized', 'performance_optimized', 'validate_config', 'version'
]

for name in expected_exports:
    assert hasattr(cqlite, name), f"Missing export: {name}"
print(f"All {len(expected_exports)} expected exports present")

CQLite version: 0.3.0
Module version: 0.3.0
All 17 expected exports present


In [2]:
# Setup paths - adjust if running from different location
# When running from bindings/python/notebooks/, data is at ../../../test-data/
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent.parent

# Use environment variable if set, otherwise use relative path
DATASETS_ROOT = Path(os.environ.get('CQLITE_DATASETS_ROOT', PROJECT_ROOT / 'test-data' / 'datasets'))
DATA_DIR = DATASETS_ROOT / 'sstables'

# Schema files
SCHEMA_BASIC = PROJECT_ROOT / 'test-data' / 'schemas' / 'basic-types.cql'
SCHEMA_COLLECTIONS = PROJECT_ROOT / 'test-data' / 'schemas' / 'collections.cql'
SCHEMA_TIMESERIES = PROJECT_ROOT / 'test-data' / 'schemas' / 'time-series.cql'
SCHEMA_WIDE_ROWS = PROJECT_ROOT / 'test-data' / 'schemas' / 'wide-rows.cql'

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Data directory exists: {DATA_DIR.exists()}")
print(f"Schema files exist: basic={SCHEMA_BASIC.exists()}, collections={SCHEMA_COLLECTIONS.exists()}")

Project root: /Users/patrick/local_projects/cqlite
Data directory: /Users/patrick/local_projects/cqlite/test-data/datasets/sstables
Data directory exists: True
Schema files exist: basic=True, collections=True


## 2. Basic Query Workflow

In [3]:
# Basic open/execute/close pattern using context manager (recommended)
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    print(f"Database opened: {db}")
    print(f"Is closed: {db.is_closed}")
    
    # Execute a simple query
    result = db.execute("SELECT * FROM test_basic.simple_table LIMIT 5")
    
    print(f"\nQuery result:")
    print(f"  Rows: {len(result.rows)}")
    print(f"  Execution time: {result.execution_time_ms}ms")
    print(f"  Rows affected: {result.rows_affected}")
    
print(f"\nAfter context exit - is_closed: {db.is_closed}")

Database opened: Database(open)
Is closed: False

Query result:
  Rows: 5
  Execution time: 7ms
  Rows affected: 5

After context exit - is_closed: True


In [4]:
# Row access patterns
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    result = db.execute("SELECT * FROM test_basic.simple_table LIMIT 3")
    
    for i, row in enumerate(result):
        print(f"\n--- Row {i} ---")
        
        # Dict-like access
        print(f"Keys: {row.keys()}")
        print(f"As dict: {row.to_dict()}")
        
        # Individual column access
        for key in list(row.keys())[:3]:  # First 3 columns
            print(f"  {key}: {row[key]} (type: {type(row[key]).__name__})")


--- Row 0 ---
Keys: ['weight', 'account_balance', 'ascii_field', 'varchar_field', 'session_id', 'description', 'ip_address', 'name', 'duration_val', 'small_number', 'height', 'medium_number', 'id', 'birth_date', 'work_time', 'created', 'age', 'salary', 'active']
As dict: {'weight': 74.45, 'account_balance': Decimal('69799.73'), 'ascii_field': 'ascii', 'varchar_field': 'realize', 'session_id': UUID('79478f1a-a251-11f0-a18d-d6726a637a4c'), 'description': b'\xd7\xfe\x1e\x8c\x1d\xf3\xa1tJOd\x04X\x1a\x8dDf\x13\x01\x86\x0ft\xcc^\x9d\xa2|\xd5\x80\x95`\xc5\x88\x13\xed\x13R\xf2\xdf\n\x95\xe80bc6\xbb\x88\xb68\xb4\x95i\x06\x82=D\xc2\x93Hw\x92\xac\x8e6\x84\xdb)\'\xbe\xdc\xa1\xd7\xce_\xba\x1c\xf92N\xdb\\\xa7P\xdd.\x1f\xecIc\x1b&Y\x12\xafd\xab\xdf\xe84\xbalk\xaa\x03S\xe1Q\x00Q\x1d_\x069\r\x1b\x1c \xdf\xf5\x16\xbd\x1a\xba\x91\xf9m\xf4\x14h\x9c8\xe1O\xda\xa2\xd1\xe2\xfe\xf3\x8c\x15;\x8b4\xdcf\xf1\x84\xaa\x13w(\xfd\xc0\x19[\xcen\xde\xdd\xbb5\xd4\x84;\x84\xaa\x18\x1c\x03Q-L\x0ey\xcd\xbeK\xd0\xd7\xf90\x

In [5]:
# Column metadata
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    result = db.execute("SELECT * FROM test_basic.simple_table LIMIT 1")
    
    print("Column Metadata:")
    print("-" * 60)
    for col in result.columns:
        print(f"  {col.position}: {col.name:20} {col.data_type:15} nullable={col.nullable}")

Column Metadata:
------------------------------------------------------------
  0: account_balance      Text            nullable=True
  1: active               Text            nullable=True
  2: age                  Text            nullable=True
  3: ascii_field          Text            nullable=True
  4: birth_date           Text            nullable=True
  5: created              Text            nullable=True
  6: description          Text            nullable=True
  7: duration_val         Text            nullable=True
  8: height               Text            nullable=True
  9: id                   Text            nullable=True
  10: ip_address           Text            nullable=True
  11: medium_number        Text            nullable=True
  12: name                 Text            nullable=True
  13: salary               Text            nullable=True
  14: session_id           Text            nullable=True
  15: small_number         Text            nullable=True
  16: varchar_field 

## 3. Type System Tour

CQLite converts CQL types to native Python types:

| CQL Type | Python Type |
|----------|-------------|
| NULL | None |
| BOOLEAN | bool |
| INT/BIGINT | int |
| FLOAT/DOUBLE | float |
| TEXT | str |
| BLOB | bytes |
| TIMESTAMP | datetime.datetime (UTC) |
| DATE | datetime.date |
| TIME | datetime.time |
| UUID | uuid.UUID |
| DECIMAL | decimal.Decimal |
| DURATION | datetime.timedelta |
| LIST | list |
| SET | frozenset |
| MAP | dict |
| TUPLE | tuple |
| UDT | dict |

In [6]:
import datetime
import uuid
from decimal import Decimal

# Primitive types from test_basic
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    result = db.execute("SELECT * FROM test_basic.simple_table LIMIT 1")
    
    if result.rows:
        row = result.rows[0]
        print("Primitive Types:")
        print("-" * 50)
        
        for key in row.keys():
            value = row[key]
            type_name = type(value).__name__ if value is not None else 'NoneType'
            display_val = repr(value)[:50] + '...' if len(repr(value)) > 50 else repr(value)
            print(f"  {key:20} = {display_val:30} ({type_name})")

Primitive Types:
--------------------------------------------------
  id                   = UUID('0023ece7-7c4e-4705-9068-d1a59ec5fe19') (UUID)
  work_time            = datetime.time(1, 12, 5, 926782) (time)
  session_id           = UUID('79478f1a-a251-11f0-a18d-d6726a637a4c') (UUID)
  weight               = 74.45                          (float)
  name                 = 'Debbie Soto'                  (str)
  birth_date           = datetime.date(2025, 2, 22)     (date)
  duration_val         = datetime.timedelta(seconds=3033) (timedelta)
  ascii_field          = 'ascii'                        (str)
  small_number         = 105                            (int)
  height               = 1.6699999570846558             (float)
  age                  = 79                             (int)
  active               = True                           (bool)
  account_balance      = Decimal('69799.73')            (Decimal)
  salary               = 152974                         (int)
  description 

In [7]:
# Temporal types from test_timeseries
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_TIMESERIES)) as db:
    result = db.execute("SELECT * FROM test_timeseries.sensor_data LIMIT 3")
    
    print("Temporal Types:")
    print("-" * 50)
    for row in result:
        for key in row.keys():
            value = row[key]
            if isinstance(value, (datetime.datetime, datetime.date, datetime.time, datetime.timedelta)):
                print(f"  {key}: {value} (type: {type(value).__name__})")

Temporal Types:
--------------------------------------------------
  timestamp: 2025-10-06 01:00:30.616000+00:00 (type: datetime)
  timestamp: 2025-10-05 13:48:21.616000+00:00 (type: datetime)
  timestamp: 2025-10-05 13:34:21.616000+00:00 (type: datetime)


In [8]:
# Collection types from test_collections
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_COLLECTIONS)) as db:
    result = db.execute("SELECT * FROM test_collections.collection_table LIMIT 3")
    
    print("Collection Types:")
    print("-" * 50)
    for i, row in enumerate(result):
        print(f"\nRow {i}:")
        for key in row.keys():
            value = row[key]
            if isinstance(value, (list, frozenset, dict, tuple)):
                type_name = type(value).__name__
                preview = str(value)[:60] + '...' if len(str(value)) > 60 else str(value)
                print(f"  {key}: {type_name} = {preview}")

Collection Types:
--------------------------------------------------

Row 0:
  numbers_set: frozenset = frozenset()
  ordered_values: list = [datetime.datetime(2025, 1, 6, 1, 12, 6, 704000, tzinfo=date...
  metadata_map: dict = {'decide': 660274, 'give': 369790, 'summer': 349795, 'town':...

Row 1:
  numbers_set: frozenset = frozenset()
  metadata_map: dict = {'every': 787799, 'goal': 392470, 'line': 12539, 'rule': 360...
  ordered_values: list = [datetime.datetime(2024, 11, 15, 1, 12, 7, 38000, tzinfo=dat...

Row 2:
  ordered_values: list = [datetime.datetime(2024, 11, 16, 1, 12, 6, 732000, tzinfo=da...
  numbers_set: frozenset = frozenset()
  metadata_map: dict = {'always': 37075, 'report': 404272, 'second': 516621, 'they'...


In [9]:
# UDT types from test_collections
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_COLLECTIONS)) as db:
    result = db.execute("SELECT * FROM test_collections.collections_with_udts LIMIT 3")
    
    print("UDT (User-Defined Types):")
    print("-" * 50)
    for row in result:
        for key, value in row.items():
            if isinstance(value, dict) and '_type' in value:
                print(f"\n{key} (UDT):")
                for k, v in value.items():
                    print(f"    {k}: {v}")

UDT (User-Defined Types):
--------------------------------------------------


## 4. Streaming for Large Data

Use `execute_streaming()` for memory-efficient processing of large result sets.

In [10]:
# Streaming with default configuration
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    count = 0
    for row in db.execute_streaming("SELECT * FROM test_basic.simple_table"):
        count += 1
    
    print(f"Streamed {count} rows with default config")

Streamed 1000 rows with default config


In [11]:
# Streaming with custom configuration
config = cqlite.StreamingConfig(buffer_size=256, chunk_size=500)
print(f"Config: buffer_size={config.buffer_size}, chunk_size={config.chunk_size}")

with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    iterator = db.execute_streaming("SELECT * FROM test_basic.simple_table", config=config)
    
    count = 0
    for row in iterator:
        count += 1
        if count % 200 == 0:
            print(f"  Processed {count} rows (received: {iterator.rows_received})")
    
    print(f"\nTotal: {count} rows")

Config: buffer_size=256, chunk_size=500
  Processed 200 rows (received: 200)
  Processed 400 rows (received: 400)
  Processed 600 rows (received: 600)
  Processed 800 rows (received: 800)
  Processed 1000 rows (received: 1000)

Total: 1000 rows


In [12]:
# Early termination (safe to break)
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    count = 0
    for row in db.execute_streaming("SELECT * FROM test_basic.simple_table"):
        count += 1
        if count >= 10:
            print(f"Breaking early after {count} rows")
            break
    
    # Resources are cleaned up automatically
    print("Cleanup successful")

Breaking early after 10 rows
Cleanup successful


## 5. Prepared Statements & Query Stats

In [13]:
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    # Prepare a query for analysis
    stmt = db.prepare("SELECT * FROM test_basic.simple_table")
    
    print("Prepared Statement:")
    print(f"  Query: {stmt.query}")
    print(f"  Parameters: {stmt.parameter_count}")
    
    # Get query plan statistics
    stats = stmt.stats()
    print(f"\nQuery Stats:")
    for key, value in stats.items():
        print(f"  {key}: {value}")

Prepared Statement:
  Query: SELECT * FROM test_basic.simple_table
  Parameters: 0

Query Stats:
  parameter_count: 0
  plan_type: TableScan
  estimated_cost: 1000.0
  estimated_rows: 100000
  cache_friendly: True


In [14]:
# Database statistics
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    # Run a query first to populate stats
    _ = db.execute("SELECT * FROM test_basic.simple_table LIMIT 10")
    
    stats = db.stats()
    
    print("Storage Stats:")
    for key, value in stats.storage_stats.items():
        if 'size' in key:
            print(f"  {key}: {value:,} bytes ({value / 1024 / 1024:.2f} MB)")
        else:
            print(f"  {key}: {value}")
    
    print("\nMemory Stats:")
    for key, value in stats.memory_stats.items():
        if 'memory' in key:
            print(f"  {key}: {value:,} bytes ({value / 1024 / 1024:.2f} MB)")
        else:
            print(f"  {key}: {value}")
    
    if stats.query_stats:
        print("\nQuery Stats:")
        for key, value in stats.query_stats.items():
            print(f"  {key}: {value}")

Storage Stats:
  sstable_count: 1
  total_size: 12,718 bytes (0.01 MB)
  total_entries: 0
  total_tables: 1
  average_size: 12,718 bytes (0.01 MB)

Memory Stats:
  block_cache_hits: 0
  block_cache_misses: 0
  row_cache_hits: 0
  row_cache_misses: 0
  total_memory_used: 0 bytes (0.00 MB)
  buffer_allocations: 0
  buffer_deallocations: 0

Query Stats:
  total_queries: 1
  error_queries: 0
  avg_execution_time_us: 8525
  cache_hit_ratio: 0.0
  rows_affected: 10


## 6. Configuration Presets

In [15]:
# Memory optimized preset (256 MB max memory)
memory_config = cqlite.memory_optimized()
print("Memory Optimized Config:")
print(f"  {memory_config}")

# Performance optimized preset (4 GB max memory)
perf_config = cqlite.performance_optimized()
print("\nPerformance Optimized Config:")
print(f"  {perf_config}")

Memory Optimized Config:
  {'storage': {'max_sstable_size': 16777216, 'memtable_size_threshold': 4194304, 'compaction': {'strategy': 'Leveled', 'max_sstables': 10, 'size_ratio': 2.0, 'max_threads': 2, 'background_interval': {'secs': 300, 'nanos': 0}, 'auto_compaction': True}, 'block_size': 65536, 'compression': {'enabled': True, 'algorithm': 'Zstd', 'level': 1, 'min_block_size': 1024}, 'enable_bloom_filters': True, 'bloom_filter_fp_rate': 0.01, 'io_threads': 4, 'sync_mode': 'Normal'}, 'memory': {'max_memory': 268435456, 'block_cache': {'enabled': True, 'max_size': 67108864, 'policy': 'Lru'}, 'row_cache': {'enabled': True, 'max_size': 33554432, 'policy': 'Lru'}, 'query_cache': {'enabled': True, 'max_size': 16777216, 'policy': 'Lru'}, 'allocator': {'use_custom': False, 'small_pool_size': 67108864, 'large_pool_size': 268435456}}, 'query': {'max_execution_time': {'secs': 300, 'nanos': 0}, 'max_result_rows': 1000000, 'plan_cache_size': 1000, 'enable_optimization': True, 'parallel': {'enable

In [16]:
# Validate custom configuration
custom_config = {
    "memory": {"max_memory": 512 * 1024 * 1024}  # 512 MB
}

is_valid = cqlite.validate_config(custom_config)
print(f"Custom config valid: {is_valid}")

# Open with config preset by name
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC), config="memory_optimized") as db:
    result = db.execute("SELECT * FROM test_basic.simple_table LIMIT 1")
    print(f"Opened with memory_optimized preset, got {len(result.rows)} row(s)")

ValueError: Invalid config dict: missing field `block_cache` at line 1 column 36

## 7. Error Handling

In [17]:
# Exception hierarchy
print("Exception Hierarchy:")
print(f"  CqliteError base: {cqlite.CqliteError.__bases__}")
print(f"  SchemaError inherits from: {cqlite.SchemaError.__bases__}")
print(f"  QueryError inherits from: {cqlite.QueryError.__bases__}")
print(f"  ParseError inherits from: {cqlite.ParseError.__bases__}")

Exception Hierarchy:
  CqliteError base: (<class 'Exception'>,)
  SchemaError inherits from: (<class 'cqlite.CqliteError'>,)
  QueryError inherits from: (<class 'cqlite.CqliteError'>,)
  ParseError inherits from: (<class 'cqlite.CqliteError'>,)


In [18]:
# Parse error - malformed query
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    try:
        db.execute("SELEKT * FORM users")
    except cqlite.ParseError as e:
        print(f"ParseError caught: {e}")
    except cqlite.CqliteError as e:
        print(f"CqliteError caught (fallback): {e}")

CqliteError caught (fallback): Query execution error: Unsupported query type: SELEKT


In [19]:
# Query error - nonexistent table
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    try:
        db.execute("SELECT * FROM nonexistent_keyspace.nonexistent_table")
    except cqlite.QueryError as e:
        print(f"QueryError caught: {e}")
    except cqlite.CqliteError as e:
        print(f"CqliteError caught (fallback): {e}")

In [20]:
# RuntimeError - using closed database
db = cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC))
db.close()

try:
    db.execute("SELECT * FROM test_basic.simple_table LIMIT 1")
except RuntimeError as e:
    print(f"RuntimeError caught: {e}")

RuntimeError caught: Database is closed


In [21]:
# Schema error - invalid schema file
try:
    db = cqlite.open(str(DATA_DIR), schema="nonexistent_schema.cql")
except (cqlite.SchemaError, IOError) as e:
    print(f"Schema/IO error caught: {type(e).__name__}: {e}")

Schema/IO error caught: SchemaError: Schema error: Schema loading failed with 1 error(s): FileRead: Failed to discover files: Invalid path: Path does not exist: nonexistent_schema.cql


## 8. All 33 Tables Smoke Test

Validate that all test tables are accessible.

In [22]:
# All 33 tables organized by keyspace
ALL_TABLES = {
    "test_basic": {
        "schema": SCHEMA_BASIC,
        "tables": [
            "simple_table",
            "composite_key_table", 
            "compression_test_table",
            "multi_partition_table",
            "ttl_test_table",
            "counters",
            "static_columns_table",
            "uncompressed_table"
        ]
    },
    "test_collections": {
        "schema": SCHEMA_COLLECTIONS,
        "tables": [
            "collection_table",
            "collection_clustering_table",
            "collections_with_udts",
            "empty_collections_table",
            "frozen_collections_table",
            "large_collections_table",
            "nested_collections_table",
            "typed_collections_table"
        ]
    },
    "test_timeseries": {
        "schema": SCHEMA_TIMESERIES,
        "tables": [
            "sensor_data",
            "event_store",
            "user_sessions",
            "app_metrics",
            "log_entries",
            "stock_prices",
            "tick_data",
            "time_bucketed_counters",
            "user_activity"
        ]
    },
    "test_wide_rows": {
        "schema": SCHEMA_WIDE_ROWS,
        "tables": [
            "wide_partition_table",
            "chat_messages",
            "document_versions",
            "large_blob_table",
            "many_columns_table",
            "multi_metric_timeseries",
            "product_catalog",
            "sparse_data_table"
        ]
    }
}

print(f"Total tables to test: {sum(len(ks['tables']) for ks in ALL_TABLES.values())}")

Total tables to test: 33


In [23]:
# Run smoke test on all tables
results = {"pass": [], "fail": [], "empty": []}

for keyspace, config in ALL_TABLES.items():
    schema = config["schema"]
    tables = config["tables"]
    
    print(f"\n{keyspace} ({len(tables)} tables):")
    print("-" * 50)
    
    with cqlite.open(str(DATA_DIR), schema=str(schema)) as db:
        for table in tables:
            try:
                result = db.execute(f"SELECT * FROM {keyspace}.{table} LIMIT 5")
                row_count = len(result.rows)
                
                if row_count > 0:
                    status = "PASS"
                    results["pass"].append(f"{keyspace}.{table}")
                else:
                    status = "EMPTY"
                    results["empty"].append(f"{keyspace}.{table}")
                
                print(f"  {table:30} {status:6} ({row_count} rows)")
                
            except Exception as e:
                status = "FAIL"
                results["fail"].append(f"{keyspace}.{table}")
                print(f"  {table:30} {status:6} ({type(e).__name__}: {str(e)[:40]})")


test_basic (8 tables):
--------------------------------------------------
  simple_table                   PASS   (5 rows)
  composite_key_table            PASS   (5 rows)
  compression_test_table         PASS   (5 rows)
  multi_partition_table          PASS   (5 rows)
  ttl_test_table                 PASS   (5 rows)
  counters                       PASS   (5 rows)
  static_columns_table           PASS   (5 rows)
  uncompressed_table             PASS   (5 rows)

test_collections (8 tables):
--------------------------------------------------
  collection_table               PASS   (5 rows)
  collection_clustering_table    PASS   (5 rows)
  collections_with_udts          PASS   (5 rows)
  empty_collections_table        PASS   (5 rows)
  frozen_collections_table       PASS   (5 rows)
  large_collections_table        PASS   (5 rows)
  nested_collections_table       PASS   (5 rows)
  typed_collections_table        PASS   (1 rows)

test_timeseries (9 tables):
-------------------------------

In [24]:
# Summary
print("\n" + "=" * 50)
print("SMOKE TEST SUMMARY")
print("=" * 50)

total = len(results["pass"]) + len(results["fail"]) + len(results["empty"])
print(f"\nTotal tables: {total}")
print(f"  PASS:  {len(results['pass']):2} ({100*len(results['pass'])/total:.0f}%)")
print(f"  EMPTY: {len(results['empty']):2} ({100*len(results['empty'])/total:.0f}%)")
print(f"  FAIL:  {len(results['fail']):2} ({100*len(results['fail'])/total:.0f}%)")

if results["fail"]:
    print(f"\nFailed tables:")
    for t in results["fail"]:
        print(f"  - {t}")

if results["empty"]:
    print(f"\nEmpty tables (may need data fetch):")
    for t in results["empty"]:
        print(f"  - {t}")


SMOKE TEST SUMMARY

Total tables: 33
  PASS:  33 (100%)
  EMPTY:  0 (0%)
  FAIL:   0 (0%)


## 9. Advanced Usage Patterns

In [25]:
# Multiple databases simultaneously (independent handles)
db1 = cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC))
db2 = cqlite.open(str(DATA_DIR), schema=str(SCHEMA_COLLECTIONS))

r1 = db1.execute("SELECT * FROM test_basic.simple_table LIMIT 1")
r2 = db2.execute("SELECT * FROM test_collections.collection_table LIMIT 1")

print(f"DB1 query: {len(r1.rows)} rows from simple_table")
print(f"DB2 query: {len(r2.rows)} rows from collection_table")

db1.close()
print(f"\nAfter closing db1: db1.is_closed={db1.is_closed}, db2.is_closed={db2.is_closed}")

# db2 still works
r3 = db2.execute("SELECT * FROM test_collections.collection_table LIMIT 1")
print(f"DB2 still works: {len(r3.rows)} rows")

db2.close()

DB1 query: 1 rows from simple_table
DB2 query: 1 rows from collection_table

After closing db1: db1.is_closed=True, db2.is_closed=False
DB2 still works: 1 rows


In [26]:
# Convert full result to various formats
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    result = db.execute("SELECT * FROM test_basic.simple_table LIMIT 3")
    
    # As list of dicts (for JSON serialization, pandas, etc.)
    rows_as_dicts = [row.to_dict() for row in result]
    print(f"Rows as list of dicts: {len(rows_as_dicts)} items")
    
    # Full result dict including metadata
    full_result = result.to_dict()
    print(f"\nFull result keys: {full_result.keys()}")
    print(f"  rows_affected: {full_result['rows_affected']}")
    print(f"  execution_time_ms: {full_result['execution_time_ms']}")
    print(f"  columns: {len(full_result['columns'])} columns")

Rows as list of dicts: 3 items

Full result keys: dict_keys(['rows', 'rows_affected', 'execution_time_ms', 'columns'])
  rows_affected: 3
  execution_time_ms: 7
  columns: 19 columns


## Acceptance Test Complete!

If all cells ran without errors and the smoke test shows mostly PASS results, the Python bindings are working correctly.

### Troubleshooting

- **Empty tables**: Run `bash test-data/scripts/fetch-datasets.sh` to download SSTable data
- **Import errors**: Rebuild bindings with `cd bindings/python && maturin develop`
- **Path errors**: Ensure `CQLITE_DATASETS_ROOT` environment variable is set correctly